# PaySim (UPI) - XGBoost Fraud Detection Training

Kaggle-ready notebook aligned to the current Phase 4A serving contract.

## 1. Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn joblib

## 2. Configuration

In [ ]:
# Set to None for full training, or a small number (for example 10000) for dry-run
SAMPLE_ROWS = 10000

ARTIFACT_VERSION = "v1"
N_ESTIMATORS = 300
MAX_DEPTH = 6
LEARNING_RATE = 0.1
TRAIN_RATIO = 0.8

DATA_PATH_CANDIDATES = [
    "/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv",
    "/kaggle/input/archive-12/PS_20174392719_1491204439457_log.csv",
    "/kaggle/input/archive-12/archive (12)/PS_20174392719_1491204439457_log.csv",
]
OUTPUT_DIR = "/kaggle/working/artifacts"

## 3. Load Data

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find PaySim CSV. Update DATA_PATH_CANDIDATES.")

read_kwargs = {"nrows": SAMPLE_ROWS} if SAMPLE_ROWS is not None else {}
df = pd.read_csv(DATA_PATH, **read_kwargs)

print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean():.6f}")
df.head()

## 4. Feature Engineering

In [ ]:
EPSILON = 1e-6
RISKY_TYPES = {"TRANSFER", "CASH_OUT"}
STEPS_24H = 24
STEPS_7D = 168

PAYSIM_TRAINING_FEATURES = [
    "type_risk_flag",
    "amount_to_orig_balance_ratio",
    "orig_balance_consistency_error",
    "dest_balance_consistency_error",
    "orig_balance_drain_pct",
    "sender_txn_count_24h",
    "sender_amount_zscore_7d",
    "sender_dest_pair_novelty",
    "dest_inbound_txn_count_24h",
]

PAYSIM_ONLINE_FEATURES = [
    "type_risk_flag",
    "amount_to_orig_balance_ratio",
    "orig_balance_consistency_error",
    "sender_txn_count_24h",
    "sender_amount_zscore_7d",
    "sender_dest_pair_novelty",
]

def build_paysim_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("step").reset_index(drop=True)

    df["type_risk_flag"] = df["type"].isin(RISKY_TYPES).astype(int)

    df["amount_to_orig_balance_ratio"] = np.clip(
        df["amount"] / np.maximum(df["oldbalanceOrg"], EPSILON), 0.0, 1.0
    )

    df["orig_balance_consistency_error"] = np.clip(
        np.abs(df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"])
        / np.maximum(df["oldbalanceOrg"], EPSILON),
        0.0,
        1.0,
    )

    is_merchant = df["nameDest"].astype(str).str.startswith("M")
    raw_dest_err = np.clip(
        np.abs(df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"])
        / np.maximum(df["amount"], EPSILON),
        0.0,
        1.0,
    )
    df["dest_balance_consistency_error"] = np.where(is_merchant, 0.0, raw_dest_err)

    is_risky = df["type"].isin(RISKY_TYPES)
    raw_drain = np.clip(
        (df["oldbalanceOrg"] - df["newbalanceOrig"])
        / np.maximum(df["oldbalanceOrg"], EPSILON),
        0.0,
        1.0,
    )
    df["orig_balance_drain_pct"] = np.where(is_risky, raw_drain, 0.0)

    n = len(df)
    sender_txn_count_24h = np.zeros(n, dtype=np.int64)
    sender_amount_zscore_7d = np.zeros(n, dtype=np.float64)
    sender_dest_pair_novelty = np.zeros(n, dtype=np.int64)
    dest_inbound_txn_count_24h = np.zeros(n, dtype=np.int64)

    sender_history = {}
    sender_dest_pairs = {}
    dest_history = {}

    for i in range(n):
        row = df.iloc[i]
        sender = row["nameOrig"]
        dest = row["nameDest"]
        step = int(row["step"])
        amount = float(row["amount"])

        hist = sender_history.get(sender, [])
        sender_txn_count_24h[i] = sum(1 for s, _ in hist if step - s <= STEPS_24H)

        amounts_7d = [a for s, a in hist if step - s <= STEPS_7D]
        if len(amounts_7d) >= 2:
            mean_v = np.mean(amounts_7d)
            std_v = np.std(amounts_7d)
            if std_v > EPSILON:
                sender_amount_zscore_7d[i] = (amount - mean_v) / std_v

        seen = sender_dest_pairs.get(sender, set())
        sender_dest_pair_novelty[i] = 0 if dest in seen else 1

        d_hist = dest_history.get(dest, [])
        dest_inbound_txn_count_24h[i] = sum(1 for s in d_hist if step - s <= STEPS_24H)

        sender_history.setdefault(sender, []).append((step, amount))
        sender_dest_pairs.setdefault(sender, set()).add(dest)
        dest_history.setdefault(dest, []).append(step)

        if i % 500000 == 0 and i > 0:
            print(f"Feature progress: {i}/{n}")

    df["sender_txn_count_24h"] = sender_txn_count_24h
    df["sender_amount_zscore_7d"] = sender_amount_zscore_7d
    df["sender_dest_pair_novelty"] = sender_dest_pair_novelty
    df["dest_inbound_txn_count_24h"] = dest_inbound_txn_count_24h

    return df[PAYSIM_TRAINING_FEATURES + ["isFraud"]].copy()

features_df = build_paysim_features(df)
print(features_df.shape)
features_df.head()

## 5. Split and Train

In [ ]:
import xgboost as xgb
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score

TARGET = "isFraud"
feature_cols = PAYSIM_TRAINING_FEATURES

split_idx = int(len(features_df) * TRAIN_RATIO)
train_df = features_df.iloc[:split_idx]
val_df = features_df.iloc[split_idx:]

X_train = train_df[feature_cols].values
y_train = train_df[TARGET].values.astype(int)
X_val = val_df[feature_cols].values
y_val = val_df[TARGET].values.astype(int)

n_neg = int(np.sum(y_train == 0))
n_pos = max(int(np.sum(y_train == 1)), 1)
scale_pos_weight = n_neg / n_pos

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("scale_pos_weight:", round(scale_pos_weight, 2))

try:
    probe_X = np.array([[0.0, 0.0], [1.0, 1.0]])
    probe_y = np.array([0, 1])
    probe = xgb.XGBClassifier(
        device="cuda",
        tree_method="hist",
        n_estimators=1,
        max_depth=1,
        eval_metric="aucpr",
        random_state=42,
    )
    probe.fit(probe_X, probe_y, verbose=False)
    device = "cuda"
except Exception as exc:
    device = "cpu"
    print("GPU unavailable, using CPU:", exc)

print("Using device:", device)

model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    scale_pos_weight=scale_pos_weight,
    device=device,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=42,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=10)

y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

metrics = {
    "threshold": 0.5,
    "precision": round(float(precision_score(y_val, y_pred, zero_division=0)), 4),
    "recall": round(float(recall_score(y_val, y_pred, zero_division=0)), 4),
    "f1": round(float(f1_score(y_val, y_pred, zero_division=0)), 4),
    "pr_auc": round(float(average_precision_score(y_val, y_pred_proba)), 4),
    "confusion_matrix": confusion_matrix(y_val, y_pred).tolist(),
}

print(json.dumps(metrics, indent=2))

## 6. Export Artifacts

In [ ]:
import json
import shutil
from datetime import datetime, timezone

import joblib

thresholds = {
    "level": {
        "low_to_medium": 0.4,
        "medium_to_high": 0.75,
    },
    "decision": {
        "allow_to_review": 0.4,
        "review_to_block": 0.9,
    },
}

manifest = {
    "domain": "paysim",
    "artifact_version": ARTIFACT_VERSION,
    "model_family": "xgboost",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "feature_order": feature_cols,
    "required_raw_fields": [
        "step",
        "type",
        "amount",
        "nameOrig",
        "oldbalanceOrg",
        "newbalanceOrig",
        "nameDest",
        "oldbalanceDest",
        "newbalanceDest",
    ],
    "online_features": PAYSIM_ONLINE_FEATURES,
    "training_features": PAYSIM_TRAINING_FEATURES,
    "score_mapping": {
        "heuristic_score": "scores.heuristic",
        "supervised_probability": "scores.supervised",
        "final_risk_score": "risk.score",
    },
    "alert_thresholds": {
        "low_to_medium": thresholds["level"]["low_to_medium"],
        "medium_to_high": thresholds["level"]["medium_to_high"],
        "allow_to_review": thresholds["decision"]["allow_to_review"],
        "review_to_block": thresholds["decision"]["review_to_block"],
    },
}

metadata = {
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "data_source": DATA_PATH,
    "sample_rows": SAMPLE_ROWS,
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "device": device,
    "n_estimators": N_ESTIMATORS,
    "max_depth": MAX_DEPTH,
    "learning_rate": LEARNING_RATE,
    "scale_pos_weight": round(scale_pos_weight, 2),
}

feature_defaults = {f: 0.0 for f in feature_cols}

version_dir = Path(OUTPUT_DIR) / "paysim" / ARTIFACT_VERSION
version_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, version_dir / "supervised_model.joblib")

for name, data in [
    ("manifest.json", manifest),
    ("feature_order.json", feature_cols),
    ("feature_defaults.json", feature_defaults),
    ("thresholds.json", thresholds),
    ("metrics.json", metrics),
    ("metadata.json", metadata),
]:
    (version_dir / name).write_text(json.dumps(data, indent=2, default=str) + "\n")

manifests_dir = Path(OUTPUT_DIR) / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(version_dir / "manifest.json", manifests_dir / "paysim_latest.json")

print("Artifacts exported to:", version_dir)
print("Latest manifest:", manifests_dir / "paysim_latest.json")